In [1]:
import os 
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd
from shapely.geometry import mapping

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    cohen_kappa_score
)
from sklearn.tree import export_text

# NEW: for local mean/std "texture" features
from scipy.ndimage import uniform_filter

# -------------------------------------------------------------------
# USER SETTINGS
# -------------------------------------------------------------------

# File paths (adjust if needed)
raster_path = r"S:\CNTH\Gus\ADU Project\fiveband_raster\new_aligned_fiveband_raster_noncornerbackyards.tif"
train_shp  = r"S:\CNTH\Gus\ADU Project\clipped_TrainingDataROIs_6_polygons.shp"
ref_shp    = r"S:\CNTH\Gus\ADU Project\clipped_ReferenceDataROIs_6_polygons.shp"

# Mask shapefile to LIMIT classification area
mask_shp   = r"N:\AdvancedActiveRS\ADU_project\ADU_RS_Space.shp"

# Output paths  (UPDATED TO V5)
classified_raster_out = (
    r"S:\CNTH\Gus\ADU Project\RF_classified_raster\fiveband_raster_noncornerbackyards_RFclass_masked_V5.tif"
)
report_out = r"S:\CNTH\Gus\ADU Project\RF_landcover_classification_report_v5.txt"

# Name of the class label field in your training & reference shapefiles
CLASS_FIELD     = "Class"
REF_CLASS_FIELD = "Class"

# Sampling parameters
MAX_SAMPLES_PER_POLY      = 5000    # max pixels to sample per training polygon
MAX_REF_SAMPLES_PER_POLY  = 5000

# Tree height threshold (in meters) for "tree >= 5 m" feature
TREE_HEIGHT_THRESHOLD = 5.0   # NOTE: height band (H) is still included as a continuous feature

# Random Forest hyperparameters
RF_N_ESTIMATORS  = 500
RF_MAX_FEATURES  = "sqrt"
RF_RANDOM_STATE  = 42

# -------------------------------------------------------------------
# 0. HELPER: NORMALIZE CLASS LABELS
# -------------------------------------------------------------------

def normalize_label(lbl):
    """
    Make training and reference class names consistent.
    E.g. 'Training_Building_Structure' and 'Ref_Building_Structure'
    both become 'Building_Structure'.
    """
    if lbl is None:
        return None
    s = str(lbl)
    s = s.replace("Training_", "").replace("Ref_", "")
    return s

# -------------------------------------------------------------------
# Helper functions for local texture features
# -------------------------------------------------------------------

def local_mean(arr, size=3):
    """
    Compute local mean in a square window of given size.
    Used to capture local context (e.g., average NDVI around each pixel).
    """
    return uniform_filter(arr, size=size)

def local_std(arr, size=3):
    """
    Compute local standard deviation in a square window.
    High std => more texture/variation (e.g., tree canopy);
    Low std  => smoother surfaces (e.g., roofs, pavement).
    """
    mean = uniform_filter(arr, size=size)
    mean_sq = uniform_filter(arr**2, size=size)
    return np.sqrt(np.maximum(mean_sq - mean**2, 0))

# -------------------------------------------------------------------
# 1. LOAD RASTER AND BUILD FEATURE STACK
#    (RGB, NIR, DEM, NDVI, GRVI + NEW TEXTURE/SHADOW FEATURES)
# -------------------------------------------------------------------

print("Loading raster and building feature stack...")

with rasterio.open(raster_path) as src:
    profile   = src.profile
    crs       = src.crs
    transform = src.transform
    nodata    = src.nodata
    height, width = src.height, src.width

    # Read all 5 bands: shape (bands, rows, cols)
    data = src.read().astype("float32")

# Assuming band order: 1=R, 2=G, 3=B, 4=NIR, 5=Height
R   = data[0]
G   = data[1]
B   = data[2]
NIR = data[3]
H   = data[4]  # digital height model

# -------------------------------------------------------------------
# Spectral indices (NDVI, GRVI, shadow index)
# -------------------------------------------------------------------

np.seterr(divide="ignore", invalid="ignore")

# NDVI: differentiates vegetation vs non-vegetation
ndvi = (NIR - R) / (NIR + R)

# GRVI: green-red vegetation index (helps with vegetation intensity)
grvi = (G - R) / (G + R)

# Shadow index: distinguishes shadowed surfaces (often darker, high B relative to R)
shadow_index = (B - R) / (B + R)

# Replace NaNs (where denominator=0) with 0 to keep everything finite
ndvi         = np.nan_to_num(ndvi, nan=0.0)
grvi         = np.nan_to_num(grvi, nan=0.0)
shadow_index = np.nan_to_num(shadow_index, nan=0.0)

# -------------------------------------------------------------------
# NEW: Local texture / context features
# -------------------------------------------------------------------
# These are designed to help:
#  - Trees vs Buildings: tree canopy is more textured (higher std) than roofs.
#  - Impervious vs Soil_NLV: impervious surfaces often smoother than patchy soil.
# We use a 3x3 window to stay small and local in backyards.

# Local mean NDVI (3x3) — background vegetation level/context
ndvi_mean_3x3 = local_mean(ndvi, size=3)

# Local NDVI standard deviation (3x3) — texture in vegetation
ndvi_std_3x3  = local_std(ndvi, size=3)

# Local NIR standard deviation (3x3) — helps separate rough trees vs smooth roofs/impervious
nir_std_3x3   = local_std(NIR, size=3)

# -------------------------------------------------------------------
# Build full feature stack:
# ORIGINAL: [R, G, B, NIR, H, NDVI, GRVI]
# NEW: add [NDVI_mean_3x3, NDVI_std_3x3, NIR_std_3x3, shadow_index]
# -------------------------------------------------------------------

features_stack = np.stack(
    [
        R, G, B,             # spectral RGB
        NIR,                 # NIR band
        H,                   # height (continuous)
        ndvi, grvi,          # vegetation indices
        ndvi_mean_3x3,       # local NDVI mean (context)
        ndvi_std_3x3,        # local NDVI std (texture)
        nir_std_3x3,         # local NIR std (texture)
        shadow_index         # shadow-related index
    ],
    axis=0
)  # shape: (n_features, rows, cols)

feature_names = [
    "R", "G", "B",
    "NIR",
    "Height",
    "NDVI", "GRVI",
    "NDVI_mean_3x3",
    "NDVI_std_3x3",
    "NIR_std_3x3",
    "Shadow_index"
]

# Base valid mask (nodata + finite checks on core indices)
# NOTE: derived features are finite if these base ones are finite.
if nodata is not None:
    base_valid_mask = (R != nodata) & np.isfinite(ndvi) & np.isfinite(grvi)
else:
    base_valid_mask = np.isfinite(ndvi) & np.isfinite(grvi)

# -------------------------------------------------------------------
# 1b. BUILD MASK FROM ADU_RS_Space.shp (ONLY CLASSIFY INSIDE)
# -------------------------------------------------------------------

print("Loading mask shapefile to restrict classification area...")

mask_gdf = gpd.read_file(mask_shp)
if mask_gdf.crs != crs:
    mask_gdf = mask_gdf.to_crs(crs)

mask_geoms = [mapping(geom) for geom in mask_gdf.geometry]

# invert=True -> True inside polygons, False outside
mask_inside = geometry_mask(
    geometries=mask_geoms,
    out_shape=(height, width),
    transform=transform,
    invert=True,
    all_touched=True
)

# Final valid mask: must be inside mask AND have valid data
valid_mask = base_valid_mask & mask_inside

# -------------------------------------------------------------------
# 2. HELPER FUNCTION TO SAMPLE PIXELS INSIDE POLYGONS
# -------------------------------------------------------------------

def sample_from_polygons(gdf, label_field, max_samples_per_poly, purpose="train"):
    """
    Samples pixels from the global features_stack inside each polygon of gdf.

    Returns X (n_samples, n_features), y (n_samples,) with normalized labels.
    """
    print(f"Sampling {purpose} data from polygons...")

    # Ensure same CRS as raster
    if gdf.crs != crs:
        gdf = gdf.to_crs(crs)

    X_list = []
    y_list = []

    for idx, row in gdf.iterrows():
        geom = [mapping(row.geometry)]
        raw_label = row[label_field]
        label = normalize_label(raw_label)

        # Create mask for pixels inside this polygon
        mask = geometry_mask(
            geometries=geom,
            out_shape=(height, width),
            transform=transform,
            invert=True,       # True => pixels *inside* polygon are True
            all_touched=True
        )

        # Only consider pixels that are inside polygon AND globally valid
        poly_mask = mask & valid_mask

        rows, cols = np.where(poly_mask)
        n_pix = rows.size

        if n_pix == 0:
            continue

        samples = features_stack[:, rows, cols].T  # (n_pix, n_features)

        # Remove any non-finite rows just in case
        finite_mask = np.all(np.isfinite(samples), axis=1)
        samples = samples[finite_mask]

        if samples.shape[0] == 0:
            continue

        # Optional random subsampling per polygon
        if samples.shape[0] > max_samples_per_poly:
            sel_idx = np.random.choice(
                samples.shape[0],
                size=max_samples_per_poly,
                replace=False
            )
            samples = samples[sel_idx]

        labels = np.full(samples.shape[0], label)

        X_list.append(samples)
        y_list.append(labels)

    if len(X_list) == 0:
        raise ValueError(f"No samples extracted for {purpose} data. "
                         f"Check geometry, CRS, and label field.")

    X = np.vstack(X_list)
    y = np.concatenate(y_list)

    print(f"  -> {purpose} samples: {X.shape[0]}")
    return X, y

# -------------------------------------------------------------------
# 3. LOAD TRAINING & REFERENCE DATA AND SAMPLE PIXELS
# -------------------------------------------------------------------

train_gdf = gpd.read_file(train_shp)
ref_gdf   = gpd.read_file(ref_shp)

X_train_all, y_train_all = sample_from_polygons(
    train_gdf,
    label_field=CLASS_FIELD,
    max_samples_per_poly=MAX_SAMPLES_PER_POLY,
    purpose="training"
)

X_ref_all, y_ref_all = sample_from_polygons(
    ref_gdf,
    label_field=REF_CLASS_FIELD,
    max_samples_per_poly=MAX_REF_SAMPLES_PER_POLY,
    purpose="reference"
)

# -------------------------------------------------------------------
# 4. SPLIT TRAINING INTO TRAIN / VALIDATION AND TRAIN RF
# -------------------------------------------------------------------

print("Splitting into training / validation sets...")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_all,
    y_train_all,
    test_size=0.2,
    random_state=RF_RANDOM_STATE,
    stratify=y_train_all
)

print("Training Random Forest classifier...")

rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_features=RF_MAX_FEATURES,
    oob_score=True,          # out-of-bag score
    n_jobs=-1,               # use all CPU cores
    class_weight="balanced", # helpful if classes are imbalanced
    random_state=RF_RANDOM_STATE
    # (Optionally you can add: min_samples_leaf=3 to smooth predictions)
)

rf.fit(X_train, y_train)
class_labels = rf.classes_  # array of normalized labels
class_to_int = {label: i + 1 for i, label in enumerate(class_labels)}

print("Class to integer mapping (for raster output):")
for lbl, code in class_to_int.items():
    print(f"  {lbl} -> {code}")

print("Random Forest training complete.")
print(f"OOB score: {rf.oob_score_:.4f}")

# -------------------------------------------------------------------
# 5. VALIDATION METRICS (HOLD-OUT FROM TRAINING POLYGONS)
# -------------------------------------------------------------------

print("Evaluating on validation set (hold-out from training ROIs)...")

y_val_pred = rf.predict(X_val)

val_acc   = accuracy_score(y_val, y_val_pred)
val_kappa = cohen_kappa_score(y_val, y_val_pred)
val_cm    = confusion_matrix(y_val, y_val_pred, labels=class_labels)
val_report = classification_report(y_val, y_val_pred, labels=class_labels)

print(f"Validation accuracy: {val_acc:.4f}")
print(f"Validation Cohen's kappa: {val_kappa:.4f}")

# -------------------------------------------------------------------
# 6. REFERENCE METRICS (USING REFERENCE POLYGONS)
# -------------------------------------------------------------------

print("Evaluating against independent reference ROIs...")

y_ref_pred = rf.predict(X_ref_all)

ref_acc   = accuracy_score(y_ref_all, y_ref_pred)
ref_kappa = cohen_kappa_score(y_ref_all, y_ref_pred)

ref_cm    = confusion_matrix(y_ref_all, y_ref_pred, labels=class_labels)
ref_report = classification_report(y_ref_all, y_ref_pred, labels=class_labels)

print(f"Reference accuracy: {ref_acc:.4f}")
print(f"Reference Cohen's kappa: {ref_kappa:.4f}")

# -------------------------------------------------------------------
# 7. VARIABLE IMPORTANCE
# -------------------------------------------------------------------

importances = rf.feature_importances_
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("Feature importances:")
print(importance_df)

# -------------------------------------------------------------------
# 7b. CONFUSION-DERIVED METRICS (USER/PRODUCER, COMMISSION/OMISSION)
# -------------------------------------------------------------------

def compute_confusion_metrics(cm, labels):
    """
    Given a confusion matrix with rows=true, cols=pred, compute:

    - Producer's accuracy (per class)
    - User's accuracy (per class)
    - Omission error (1 - producer)
    - Commission error (1 - user)
    Plus row-normalized % confusion matrix.
    """
    cm = cm.astype(float)
    total    = cm.sum()
    row_sums = cm.sum(axis=1)
    col_sums = cm.sum(axis=0)

    # Overall accuracy (should match accuracy_score)
    overall_acc = np.trace(cm) / total if total > 0 else np.nan

    # Avoid zero-division
    row_sums_safe = np.where(row_sums == 0, np.nan, row_sums)
    col_sums_safe = np.where(col_sums == 0, np.nan, col_sums)

    producer_acc = np.diag(cm) / row_sums_safe  # true -> predicted correctly
    user_acc     = np.diag(cm) / col_sums_safe  # predicted -> actually correct

    omission_error   = 1.0 - producer_acc
    commission_error = 1.0 - user_acc

    # Row-normalized confusion matrix (%)
    row_norm = cm / row_sums_safe[:, None] * 100.0

    metrics_df = pd.DataFrame({
        "Class": labels,
        "Reference_Pixels": row_sums.astype(int),
        "Classified_Pixels": col_sums.astype(int),
        "Producer_Accuracy_%": producer_acc * 100.0,
        "User_Accuracy_%": user_acc * 100.0,
        "Omission_Error_%": omission_error * 100.0,
        "Commission_Error_%": commission_error * 100.0
    })

    return overall_acc, metrics_df, row_norm

val_overall_from_cm, val_metrics_df, val_row_norm = compute_confusion_metrics(val_cm, class_labels)
ref_overall_from_cm, ref_metrics_df, ref_row_norm = compute_confusion_metrics(ref_cm, class_labels)

# DataFrames for confusion matrices (counts & row-normalized %)
val_cm_df = pd.DataFrame(
    val_cm,
    index=[f"true_{c}" for c in class_labels],
    columns=[f"pred_{c}" for c in class_labels]
)

val_cm_pct_df = pd.DataFrame(
    np.round(val_row_norm, 2),
    index=[f"true_{c}" for c in class_labels],
    columns=[f"pred_{c}" for c in class_labels]
)

ref_cm_df = pd.DataFrame(
    ref_cm,
    index=[f"true_{c}" for c in class_labels],
    columns=[f"pred_{c}" for c in class_labels]
)

ref_cm_pct_df = pd.DataFrame(
    np.round(ref_row_norm, 2),
    index=[f"true_{c}" for c in class_labels],
    columns=[f"pred_{c}" for c in class_labels]
)

# -------------------------------------------------------------------
# 7c. SAMPLE TREE STRUCTURE FROM THE RANDOM FOREST
# -------------------------------------------------------------------

print("Exporting a sample decision tree (first estimator)...")
tree_text = export_text(
    rf.estimators_[0],
    feature_names=feature_names,
    max_depth=4  # limit depth for readability in the text report
)

# -------------------------------------------------------------------
# 8. WRITE REPORT TO TEXT FILE
# -------------------------------------------------------------------

print(f"Writing diagnostics report to: {report_out}")

with open(report_out, "w", encoding="utf-8") as f:
    f.write("Random Forest Land Cover Classification Report (v5)\n")
    f.write("=================================================\n\n")
    f.write(f"Input raster: {raster_path}\n")
    f.write(f"Training shapefile: {train_shp}\n")
    f.write(f"Reference shapefile: {ref_shp}\n")
    f.write(f"Mask shapefile (classification extent): {mask_shp}\n\n")

    f.write("Random Forest Parameters:\n")
    f.write(f"  n_estimators      : {RF_N_ESTIMATORS}\n")
    f.write(f"  max_features      : {RF_MAX_FEATURES}\n")
    f.write(f"  class_weight      : balanced\n")
    f.write(f"  random_state      : {RF_RANDOM_STATE}\n")
    f.write(f"  OOB score         : {rf.oob_score_:.4f}\n\n")

    f.write("Feature Names and Importances:\n")
    f.write(importance_df.to_string(index=False))
    f.write("\n\n")

    # ---------------- Validation section ----------------
    f.write("VALIDATION METRICS (hold-out from training ROIs)\n")
    f.write("-------------------------------------------------\n")
    f.write("Confusion matrix is Reference (rows=true) vs Classified (cols=predicted).\n\n")
    f.write(f"Overall Accuracy (accuracy_score) : {val_acc:.4f}\n")
    f.write(f"Cohen's kappa                     : {val_kappa:.4f}\n")
    f.write(f"Overall Accuracy (from confusion) : {val_overall_from_cm:.4f}\n\n")

    f.write("Validation Confusion Matrix (counts, rows=true, cols=pred):\n")
    f.write(val_cm_df.to_string())
    f.write("\n\n")

    f.write("Validation Confusion Matrix (row-normalized %, rows=true, cols=pred):\n")
    f.write(val_cm_pct_df.to_string())
    f.write("\n\n")

    f.write("Validation Per-Class Accuracy Metrics:\n")
    f.write(val_metrics_df.round(2).to_string(index=False))
    f.write("\n\n")

    f.write("Validation Classification Report (precision/recall/F1):\n")
    f.write(val_report)
    f.write("\n\n")

    # ---------------- Reference section ----------------
    f.write("REFERENCE METRICS (independent reference ROIs)\n")
    f.write("----------------------------------------------\n")
    f.write("Confusion matrix is Reference (rows=true) vs Classified (cols=predicted).\n\n")
    f.write(f"Overall Accuracy (accuracy_score) : {ref_acc:.4f}\n")
    f.write(f"Cohen's kappa                     : {ref_kappa:.4f}\n")
    f.write(f"Overall Accuracy (from confusion) : {ref_overall_from_cm:.4f}\n\n")

    f.write("Reference Confusion Matrix (counts, rows=true, cols=pred):\n")
    f.write(ref_cm_df.to_string())
    f.write("\n\n")

    f.write("Reference Confusion Matrix (row-normalized %, rows=true, cols=pred):\n")
    f.write(ref_cm_pct_df.to_string())
    f.write("\n\n")

    f.write("Reference Per-Class Accuracy Metrics:\n")
    f.write(ref_metrics_df.round(2).to_string(index=False))
    f.write("\n\n")

    f.write("Reference Classification Report (precision/recall/F1):\n")
    f.write(ref_report)
    f.write("\n\n")

    # ---------------- Tree structure ----------------
    f.write("Sample Decision Tree Structure (first tree in the forest)\n")
    f.write("--------------------------------------------------------\n")
    f.write("Note: max_depth limited in export_text for readability.\n\n")
    f.write(tree_text)
    f.write("\n")

# Use a single core for prediction to reduce memory overhead
rf.set_params(n_jobs=1)

# -------------------------------------------------------------------
# 9. CLASSIFY FULL RASTER IN CHUNKS AND WRITE OUTPUT TIFF
# -------------------------------------------------------------------

from rasterio.windows import Window

print("Classifying full raster in chunks to avoid memory errors...")

# Output profile
out_profile = profile.copy()
out_profile.update(
    dtype=rasterio.uint8,
    count=1,
    nodata=0  # 0 = no data / unclassified
)

# Decide how many pixels per chunk (tune if needed)
# e.g., target ~1,000,000 pixels per chunk
target_pixels_per_chunk = 1_000_000
chunk_rows = max(1, int(target_pixels_per_chunk / width))

print(f"  Image size: {height} rows x {width} cols")
print(f"  Using chunk size: {chunk_rows} rows (~{chunk_rows*width} pixels)")

with rasterio.open(classified_raster_out, "w", **out_profile) as dst:
    # Initialize output with zeros (nodata)
    dst.write(np.zeros((height, width), dtype="uint8"), 1)

    # Loop over row-chunks
    for row_start in range(0, height, chunk_rows):
        row_end   = min(row_start + chunk_rows, height)
        rows_this = row_end - row_start

        print(f"  Processing rows {row_start} to {row_end-1}...")

        # Slice features and mask for this chunk
        feats_chunk = features_stack[:, row_start:row_end, :]    # (n_features, rows_this, width)
        mask_chunk  = valid_mask[row_start:row_end, :]           # (rows_this, width)

        # Flatten to 2D
        feats_flat = feats_chunk.reshape(feats_chunk.shape[0], -1).T  # (rows_this*width, n_features)
        mask_flat  = mask_chunk.reshape(-1)

        # Valid + finite
        finite_flat  = np.all(np.isfinite(feats_flat), axis=1)
        valid_predict = mask_flat & finite_flat

        X_chunk = feats_flat[valid_predict]

        # If no valid pixels in chunk, skip
        if X_chunk.shape[0] == 0:
            continue

        # Predict for this chunk (normalized string labels)
        y_chunk_pred = rf.predict(X_chunk)

        # Map string labels to integer codes
        y_chunk_int = np.array(
            [class_to_int[label] for label in y_chunk_pred],
            dtype="uint8"
        )

        # Build classified chunk (full size for block)
        classified_flat = np.zeros(feats_flat.shape[0], dtype="uint8")
        classified_flat[valid_predict] = y_chunk_int

        classified_chunk = classified_flat.reshape(rows_this, width)

        # Define window for writing
        window = Window(
            col_off=0,
            row_off=row_start,
            width=width,
            height=rows_this
        )

        # Write this chunk into output raster
        dst.write(classified_chunk, 1, window=window)

print(f"Done. Classified raster written to: {classified_raster_out}")


Loading raster and building feature stack...
Loading mask shapefile to restrict classification area...
Sampling training data from polygons...
  -> training samples: 20156
Sampling reference data from polygons...
  -> reference samples: 23898
Splitting into training / validation sets...
Training Random Forest classifier...
Class to integer mapping (for raster output):
  Building_Structure -> 1
  ILV -> 2
  Impervious -> 3
  Soil_NLV -> 4
  Trees -> 5
  Water -> 6
Random Forest training complete.
OOB score: 0.9874
Evaluating on validation set (hold-out from training ROIs)...
Validation accuracy: 0.9876
Validation Cohen's kappa: 0.9827
Evaluating against independent reference ROIs...
Reference accuracy: 0.7916
Reference Cohen's kappa: 0.7355
Feature importances:
          feature  importance
4          Height    0.285304
10   Shadow_index    0.142492
6            GRVI    0.141516
2               B    0.139714
1               G    0.075663
3             NIR    0.065037
0               R  